#### Books:
 Fix Publication years
 Handle missing publishers/authors
 Remove duplicate ISBNs

 #### Users:
 Handle missing ages
 Remove unrealistic ages

 #### Ratings:
 Separate explicit and implicit ratings
 Remove ratings with missing books/users
 Check duplicates


In [24]:
import pandas as pd
import numpy as np

In [25]:
books = pd.read_csv("../data/Books.csv")

users = pd.read_csv("../data/Users.csv")

ratings = pd.read_csv("../data/Ratings.csv")

C:\Users\sapko\AppData\Local\Temp\ipykernel_5900\1413299316.py:1: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv("../data/Books.csv")


In [26]:
books.rename(columns={
    "Book-Title": "Book_Title",
    "Book-Author": "Book_Author",
    "Year-Of-Publication": "Year_Of_Publication",
    "Image-URL-S": "Image_URL_S",
    "Image-URL-M": "Image_URL_M",
    "Image-URL-L": "Image_URL_L"
}, inplace=True)

users.rename(columns={
    "User-ID": "User_ID"
}, inplace=True)

ratings.rename(columns={
    "User-ID": "User_ID",
    "Book-Rating": "Book_Rating"
}, inplace=True)

In [27]:
books_clean = books.copy()
users_clean = users.copy()
ratings_clean = ratings.copy()

In [28]:
books_clean.isnull().sum()

ISBN                   0
Book_Title             0
Book_Author            2
Year_Of_Publication    0
Publisher              2
Image_URL_S            0
Image_URL_M            0
Image_URL_L            3
dtype: int64

In [29]:
users_clean.isnull().sum()

User_ID          0
Location         0
Age         110762
dtype: int64

In [30]:
ratings_clean.isnull().sum()

User_ID        0
ISBN           0
Book_Rating    0
dtype: int64

In [31]:
books_clean[books_clean["Book_Author"].isnull()]

,ISBN,Book_Title,Book_Author,Year_Of_Publication,Publisher,Image_URL_S,Image_URL_M,Image_URL_L
118033,0751352497,A+ Quiz Masters:01 Earth,NaN,1999,Dorling Kindersley,http://images.amazon.com/images/P/0751352497.0...,http://images.amazon.com/images/P/0751352497.0...,http://images.amazon.com/images/P/0751352497.0...
187689,9627982032,The Credit Suisse Guide to Managing Your Perso...,NaN,1995,Edinburgh Financial Publishing,http://images.amazon.com/images/P/9627982032.0...,http://images.amazon.com/images/P/9627982032.0...,http://images.amazon.com/images/P/9627982032.0...


In [32]:
books_clean["Book_Author"] = books_clean["Book_Author"].fillna("Unknown")

In [33]:
books_clean[books_clean["Publisher"].isnull()]

,ISBN,Book_Title,Book_Author,Year_Of_Publication,Publisher,Image_URL_S,Image_URL_M,Image_URL_L
128890,193169656X,Tyrant Moon,Elaine Corvidae,2002,NaN,http://images.amazon.com/images/P/193169656X.0...,http://images.amazon.com/images/P/193169656X.0...,http://images.amazon.com/images/P/193169656X.0...
129037,1931696993,Finders Keepers,Linnea Sinclair,2001,NaN,http://images.amazon.com/images/P/1931696993.0...,http://images.amazon.com/images/P/1931696993.0...,http://images.amazon.com/images/P/1931696993.0...


In [34]:
books_clean["Publisher"] = books_clean["Publisher"].fillna("Unknown")

In [35]:
books_clean[books_clean["Image_URL_L"].isnull()]

,ISBN,Book_Title,Book_Author,Year_Of_Publication,Publisher,Image_URL_S,Image_URL_M,Image_URL_L
209538,078946697X,"DK Readers: Creating the X-Men, How It All Beg...",2000,DK Publishing Inc,http://images.amazon.com/images/P/078946697X.0...,http://images.amazon.com/images/P/078946697X.0...,http://images.amazon.com/images/P/078946697X.0...,NaN
220731,2070426769,"Peuple du ciel, suivi de 'Les Bergers\"";Jean-M...",2003,Gallimard,http://images.amazon.com/images/P/2070426769.0...,http://images.amazon.com/images/P/2070426769.0...,http://images.amazon.com/images/P/2070426769.0...,NaN
221678,0789466953,"DK Readers: Creating the X-Men, How Comic Book...",2000,DK Publishing Inc,http://images.amazon.com/images/P/0789466953.0...,http://images.amazon.com/images/P/0789466953.0...,http://images.amazon.com/images/P/0789466953.0...,NaN


In [36]:
users_clean["Age"].describe()

count    168096.000000
mean         34.751434
std          14.428097
min           0.000000
25%          24.000000
50%          32.000000
75%          44.000000
max         244.000000
Name: Age, dtype: float64

In [37]:
invalid_age = users_clean[
    (users_clean["Age"] < 5) |
    (users_clean["Age"] > 100)
]

print(f"Invalid ages: {len(invalid_age)}")
invalid_age.head()

Invalid ages: 1248


,User_ID,Location,Age
219,220,"bogota, bogota, colombia",0.0
469,470,"indianapolis, indiana, usa",0.0
561,562,"adfdaf, australian capital territory, albania",0.0
612,613,"ankara, n/a, turkey",1.0
670,671,"jeddah, jeddah, saudi arabia",1.0


In [38]:
users_clean.loc[
    (users_clean["Age"] < 5) |
    (users_clean["Age"] > 100),
    "Age"
] = np.nan

In [39]:
users_clean["Age"].describe()

count    166848.000000
mean         34.746638
std          13.633051
min           5.000000
25%          24.000000
50%          32.000000
75%          44.000000
max         100.000000
Name: Age, dtype: float64

In [40]:
users_clean["Age"].isnull().sum()

np.int64(112010)

In [41]:
# fill missing age with median

users_clean["Age"] = users_clean["Age"].fillna(
    users_clean["Age"].median()
)

In [42]:
print(users_clean["Age"].describe())
print(users_clean["Age"].isnull().sum())

count    278858.000000
mean         33.643385
std          10.630979
min           5.000000
25%          29.000000
50%          32.000000
75%          35.000000
max         100.000000
Name: Age, dtype: float64
0


In [43]:
books_clean["Year_Of_Publication"] = pd.to_numeric(
    books_clean["Year_Of_Publication"],
    errors="coerce"
)

In [44]:
books_clean["Year_Of_Publication"].describe()

count    271357.000000
mean       1959.760817
std         257.994226
min           0.000000
25%        1989.000000
50%        1995.000000
75%        2000.000000
max        2050.000000
Name: Year_Of_Publication, dtype: float64

In [45]:
invalid_years = books_clean[
    (books_clean["Year_Of_Publication"] < 1800) |
    (books_clean["Year_Of_Publication"] > 2026)
]

print(f"Invalid publication years: {len(invalid_years)}")

invalid_years["Year_Of_Publication"].value_counts().sort_index()

Invalid publication years: 4631


Year_Of_Publication
0.0       4618
1376.0       1
1378.0       1
2030.0       7
2037.0       1
2038.0       1
2050.0       2
Name: count, dtype: int64

In [46]:
# replace invalid years with NaN

books_clean.loc[
    (books_clean["Year_Of_Publication"] < 1800) |
    (books_clean["Year_Of_Publication"] > 2026),
    "Year_Of_Publication"
] = np.nan

In [47]:
# fill invalid years with median

books_clean["Year_Of_Publication"] = books_clean[
    "Year_Of_Publication"
].fillna(
    books_clean["Year_Of_Publication"].median()
)

In [48]:
print(books_clean["Year_Of_Publication"].describe())
print(books_clean["Year_Of_Publication"].isnull().sum())

count    271360.000000
mean       1993.732094
std           8.084153
min        1806.000000
25%        1989.000000
50%        1996.000000
75%        2000.000000
max        2026.000000
Name: Year_Of_Publication, dtype: float64
0


In [49]:
ratings_clean.duplicated().sum()

np.int64(0)

In [ ]:
ratings_clean.isnull().sum()

User_ID        0
ISBN           0
Book_Rating    0
dtype: int64

In [52]:
invalid_books = ~ratings_clean["ISBN"].isin(books_clean["ISBN"])

print("Invalid ISBN ratings:", invalid_books.sum())

Invalid ISBN ratings: 0


In [51]:
ratings_clean = ratings_clean[
    ratings_clean["ISBN"].isin(books_clean["ISBN"])
]

In [53]:
invalid_users = ~ratings_clean["User_ID"].isin(users_clean["User_ID"])

print("Invalid User ratings:", invalid_users.sum())

Invalid User ratings: 0


Separate Explicit and Implicit ratings

In [55]:
explicit_ratings = ratings_clean[
    ratings_clean["Book_Rating"] > 0
]

implicit_ratings = ratings_clean[
    ratings_clean["Book_Rating"] == 0
]

In [57]:
print(f"Total Ratings: {len(ratings_clean):,}")
print(f"Explicit Ratings: {len(explicit_ratings):,}")
print(f"Implicit Ratings: {len(implicit_ratings):,}")

Total Ratings: 1,031,136
Explicit Ratings: 383,842
Implicit Ratings: 647,294


In [58]:
books_clean.to_csv("../data/books_clean.csv", index=False)
users_clean.to_csv("../data/users_clean.csv", index=False)
ratings_clean.to_csv("../data/ratings_clean.csv", index=False)

In [59]:
explicit_ratings.to_csv(
    "../data/explicit_ratings.csv",
    index=False
)

implicit_ratings.to_csv(
    "../data/implicit_ratings.csv",
    index=False
)